# Kaggle — 5 phương pháp LCTA (resize) với mBERT và mT5

**Input (Kaggle Dataset):** `/kaggle/input/datasets/tuyennguyen21pt/dataset-apc/converted_apc/{train,dev,test}.apc`
(nếu Kaggle mount ở đường dẫn khác, cell 4 tự dò tìm trong `/kaggle/input`).

**Pipeline:** ATE huấn luyện 1 lần với `google/mt5-base` → APC chạy đúng **5 method** × **2 backbone** (mBERT, mT5) = **10 run**.

**5 method (đúng config trong `common/run_multiseed.py`):**

| config_id | display | LCF | CDM/CDW | ToMe merge | resize |
|---|---|---|---|---|---|
| `lcf_bip_cdm` | BiToMe+CDM | ✔ | CDM | bipartite | ✔ |
| `lcf_seq_cdm` | SLM+CDM | ✔ | CDM | sequential_local | ✔ |
| `lcf_seq_cdw` | SLM+CDW | ✔ | CDW | sequential_local | ✔ |
| `lcf_scm_cdm` | SCM+CDM | ✔ | CDM | sequential_cosine | ✔ |
| `lcf_scm_cdw` | SCM+CDW | ✔ | CDW | sequential_cosine | ✔ |

**Output (tất cả nằm trong `/kaggle/working/outputs/`):**

```
/kaggle/working/outputs/
├── 00_dataset/              train.apc, dev.apc, test.apc (+ dataset_apc.zip)
├── 01_ate/        seed_42/test_predictions.csv, ate_summary.* , checkpoints/
└── 02_apc_5_methods/        results_raw.csv, results_aggregated.csv,
                             results_summary.txt, thesis_tables.txt,
                             evaluation_precision_recall_f1.csv,
                             <backbone>/<config_id>/seed_42/best_model.pt
```


In [ ]:
import os
import sys
import subprocess
import zipfile
from pathlib import Path

REPO_URL = 'https://github.com/hotuyen21pt/KLTN-Token-Merging.git'
BRANCH = 'tuyen'
REPO_NAME = 'KLTN-Token-Merging'
ON_KAGGLE = Path('/kaggle').exists()
WORK_ROOT = Path('/kaggle/working') if ON_KAGGLE else Path.cwd()
REPO_DIR = WORK_ROOT / REPO_NAME

# ── INPUT ────────────────────────────────────────────────────────────────────
# Thư mục Kaggle Dataset chứa train.apc / dev.apc / test.apc.
DATA_INPUT_DIR = '/kaggle/input/datasets/tuyennguyen21pt/dataset-apc/converted_apc'
# Dự phòng: Kaggle thường mount dataset ở /kaggle/input/<slug>/...
DATA_INPUT_FALLBACKS = [
    '/kaggle/input/dataset-apc/converted_apc',
    '/kaggle/input/dataset-apc',
    '/kaggle/input/converted_apc',
]
# Tuỳ chọn: dùng zip thay cho thư mục (để None nếu dùng DATA_INPUT_DIR).
DATA_INPUT_ZIP = None
# Tuỳ chọn: ATE CSV có sẵn. Để None thì cell ATE sẽ tự huấn luyện mT5-base.
ATE_INPUT_CSV = None

# ── OUTPUT (mọi kết quả đều nằm dưới /kaggle/working/outputs) ────────────────
OUTPUT_ROOT = WORK_ROOT / 'outputs'
DATA_OUTPUT_DIR = OUTPUT_ROOT / '00_dataset'          # bản sao 3 file .apc đã dùng
ATE_RUNS_DIR = OUTPUT_ROOT / '01_ate'       # dự đoán ATE + metrics
ATE_CKPT_DIR = ATE_RUNS_DIR / 'checkpoints'           # checkpoint ATE
RUNS_DIR = OUTPUT_ROOT / '02_apc_5_methods'           # kết quả APC (5 method x 2 backbone)
for _dir in (OUTPUT_ROOT, DATA_OUTPUT_DIR, ATE_RUNS_DIR, ATE_CKPT_DIR, RUNS_DIR):
    _dir.mkdir(parents=True, exist_ok=True)

# ── Siêu tham số ─────────────────────────────────────────────────────────────
SEED = 42
MAX_EPOCHS = 15
ATE_EPOCHS = 20
ATE_MODEL_NAME = 'google/mt5-base'
APC_MODELS = {
    'mbert': 'bert-base-multilingual-cased',
    'mt5': 'google/mt5-base',
}

# ── Đúng 5 method, đúng config ───────────────────────────────────────────────
# (use_lcf, use_cdm, use_tome, tome_resize, merge_strategy, use_pre_tome, display_name, config_id)
# use_cdm=True  -> CDM (Context Dynamic Mask)
# use_cdm=False -> CDW (Context Dynamic Weight)
METHOD_CONFIGS = [
    (True, True,  True, True, 'bipartite',         False, 'BiToMe+CDM', 'lcf_bip_cdm'),
    (True, True,  True, True, 'sequential_local',  False, 'SLM+CDM',    'lcf_seq_cdm'),
    (True, False, True, True, 'sequential_local',  False, 'SLM+CDW',    'lcf_seq_cdw'),
    (True, True,  True, True, 'sequential_cosine', False, 'SCM+CDM',    'lcf_scm_cdm'),
    (True, False, True, True, 'sequential_cosine', False, 'SCM+CDW',    'lcf_scm_cdw'),
]
assert len(METHOD_CONFIGS) == 5, 'Phải đúng 5 method'
METHODS = {cfg[7]: cfg[6] for cfg in METHOD_CONFIGS}
assert len(METHODS) == 5, 'config_id bị trùng'

CACHE_DIR = Path('/kaggle/temp/hf') if ON_KAGGLE else WORK_ROOT / '.hf'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(CACHE_DIR)
os.environ['TRANSFORMERS_CACHE'] = str(CACHE_DIR)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['MPLBACKEND'] = 'Agg'
# Giảm phân mảnh VRAM (phải đặt trước khi import torch).
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

print('Input dir :', DATA_INPUT_DIR or '(none)')
print('Input zip :', DATA_INPUT_ZIP or '(none)')
print('Output    :', OUTPUT_ROOT)
print('  dataset ->', DATA_OUTPUT_DIR)
print('  ATE     ->', ATE_RUNS_DIR)
print('  APC     ->', RUNS_DIR)
print('ATE model :', ATE_MODEL_NAME)
print('APC models:', APC_MODELS)
for cfg in METHOD_CONFIGS:
    print(f'  {cfg[7]:<14} {cfg[6]:<12} merge={cfg[4]:<18} {"CDM" if cfg[1] else "CDW"} resize={cfg[3]}')
print('Tổng số run APC:', len(METHODS) * len(APC_MODELS))

In [ ]:
def sh(*args, cwd=None, check=True):
    cmd = [str(x) for x in args]
    print('$ ' + ' '.join(cmd), flush=True)
    return subprocess.run(cmd, cwd=cwd, check=check).returncode

if (REPO_DIR / '.git').is_dir():
    sh('git', 'fetch', '--all', '--prune', cwd=REPO_DIR)
    sh('git', 'checkout', '-B', BRANCH, f'origin/{BRANCH}', cwd=REPO_DIR)
    sh('git', 'reset', '--hard', f'origin/{BRANCH}', cwd=REPO_DIR)
else:
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    sh('git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, REPO_DIR)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
sh(sys.executable, '-m', 'pip', 'install', '-q', 'python-Levenshtein>=0.25.0', 'seaborn', 'tqdm')

In [ ]:
# Download all model files before training.
# mT5-base is used by ATE and APC; mBERT is used by APC.
from huggingface_hub import snapshot_download

for model_name in sorted(set(APC_MODELS.values()) | {ATE_MODEL_NAME}):
    print('Downloading complete snapshot:', model_name)
    snapshot_path = snapshot_download(
        repo_id=model_name,
        cache_dir=CACHE_DIR,
        resume_download=True,
    )
    print('Ready:', snapshot_path)
print('All required model files are cached.')

In [ ]:
# Tìm thư mục chứa đủ train.apc / dev.apc / test.apc.
REQUIRED_FILES = ('train.apc', 'dev.apc', 'test.apc')


def _has_all_apc(directory):
    return all((Path(directory) / name).is_file() for name in REQUIRED_FILES)


def _resolve_data_dir():
    # 1) Zip do người dùng chỉ định.
    if DATA_INPUT_ZIP:
        zip_path = Path(DATA_INPUT_ZIP)
        if not zip_path.is_file():
            raise FileNotFoundError(f'DATA_INPUT_ZIP không tồn tại: {zip_path}')
        extracted_root = WORK_ROOT / 'dataset_apc_extracted'
        extracted_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path) as archive:
            archive.extractall(extracted_root)
        found = [p.parent for p in extracted_root.rglob('train.apc') if _has_all_apc(p.parent)]
        if not found:
            raise FileNotFoundError(f'Zip phải chứa {REQUIRED_FILES} trong cùng một thư mục.')
        return found[0], f'zip: {zip_path}'

    # 2) Đường dẫn chính + các đường dẫn dự phòng.
    for candidate in [DATA_INPUT_DIR, *DATA_INPUT_FALLBACKS]:
        if candidate and _has_all_apc(candidate):
            return Path(candidate), 'đường dẫn cấu hình'

    # 3) Tự dò trong /kaggle/input (phòng khi Kaggle mount ở slug khác).
    input_root = Path('/kaggle/input')
    if input_root.is_dir():
        found = sorted(p.parent for p in input_root.rglob('train.apc') if _has_all_apc(p.parent))
        if found:
            return found[0], 'tự dò trong /kaggle/input'

    # 4) Dataset đi kèm repo.
    if _has_all_apc(REPO_DIR / 'dataset'):
        return REPO_DIR / 'dataset', 'dataset trong repo'

    searched = [DATA_INPUT_DIR, *DATA_INPUT_FALLBACKS, str(input_root), str(REPO_DIR / 'dataset')]
    raise FileNotFoundError(
        'Không tìm thấy đủ train.apc / dev.apc / test.apc. Đã tìm ở:\n  '
        + '\n  '.join(str(s) for s in searched if s)
        + '\nHãy Add Data cho notebook rồi chỉnh lại DATA_INPUT_DIR ở cell 1.'
    )


DATA_DIR, DATA_SOURCE = _resolve_data_dir()
print(f'Dataset ({DATA_SOURCE}): {DATA_DIR}')
for name in REQUIRED_FILES:
    f = DATA_DIR / name
    print(f'  {name:<10} {f.stat().st_size / 1024:10.1f} KB  {f}')

SUPPLEMENT_DIR = DATA_DIR / 'supplement'
print('Supplement:', SUPPLEMENT_DIR if SUPPLEMENT_DIR.is_dir() else '(không có — không dùng)')

ATE_CSV = Path(ATE_INPUT_CSV) if ATE_INPUT_CSV else ATE_RUNS_DIR / f'seed_{SEED}' / 'test_predictions.csv'
print('ATE CSV   :', ATE_CSV if ATE_CSV.is_file() else f'{ATE_CSV} (chưa có — cell ATE sẽ tạo)')

In [ ]:
# Sao chép đúng 3 file .apc đã dùng sang output để lưu vết (00_dataset).
import shutil

for name in REQUIRED_FILES:
    shutil.copy2(DATA_DIR / name, DATA_OUTPUT_DIR / name)

dataset_zip = DATA_OUTPUT_DIR / 'dataset_apc.zip'
with zipfile.ZipFile(dataset_zip, 'w', zipfile.ZIP_DEFLATED) as archive:
    for name in REQUIRED_FILES:
        archive.write(DATA_OUTPUT_DIR / name, name)

(DATA_OUTPUT_DIR / 'SOURCE.txt').write_text(
    f'source_dir: {DATA_DIR}\nresolved_by: {DATA_SOURCE}\nseed: {SEED}\n',
    encoding='utf-8',
)

print('Dataset output ->', DATA_OUTPUT_DIR)
for name in (*REQUIRED_FILES, 'dataset_apc.zip', 'SOURCE.txt'):
    f = DATA_OUTPUT_DIR / name
    print(f'  {name:<18} {f.stat().st_size / 1024:10.1f} KB')

In [ ]:
# Giai đoạn 1: huấn luyện ATE mT5-base một lần và dự đoán aspect term trên test.
# Output -> outputs/01_ate/seed_<SEED>/test_predictions.csv
import gc

import torch
import common.run_multiseed_ate as ate_runner


def free_vram(tag):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        free_b, total_b = torch.cuda.mem_get_info()
        print(f'[VRAM {tag}] trống {free_b / 2**30:.2f} / {total_b / 2**30:.2f} GB')

USE_SEED_PAIRED_ATE = False

if ATE_INPUT_CSV:
    ATE_CSV = Path(ATE_INPUT_CSV)
    if not ATE_CSV.is_file():
        raise FileNotFoundError(f'ATE_INPUT_CSV không tồn tại: {ATE_CSV}')
else:
    ate_args = [
        '--seeds', str(SEED),
        '--model-name', ATE_MODEL_NAME,
        '--epochs', str(ATE_EPOCHS),
        '--lr', '3e-4',
        '--data-dir', str(DATA_DIR),
        '--runs-ate-dir', str(ATE_RUNS_DIR),
        '--ckpt-dir', str(ATE_CKPT_DIR),
        '--resume',
    ]
    print('ATE command: python common/run_multiseed_ate.py ' + ' '.join(ate_args))
    sys.argv = ['run_multiseed_ate.py', *ate_args]
    ate_runner.main()
    ATE_CSV = ATE_RUNS_DIR / f'seed_{SEED}' / 'test_predictions.csv'
    USE_SEED_PAIRED_ATE = True

# Trả toàn bộ VRAM của mô hình ATE trước khi sang giai đoạn APC.
free_vram('sau ATE')

if not ATE_CSV.is_file():
    raise FileNotFoundError(f'Không tạo được dự đoán ATE: {ATE_CSV}')
print('ATE mT5-base sẵn sàng:', ATE_CSV)

In [ ]:
import torch
import common.run_multiseed as runner

# Đăng ký 2 backbone APC và ép runner chỉ chạy đúng 5 config đã chọn.
runner.MODEL_REGISTRY.update(APC_MODELS)
runner.NUM_EPOCHS = MAX_EPOCHS
runner.TRAIN_APC = DATA_DIR / 'train.apc'
runner.DEV_APC = DATA_DIR / 'dev.apc'
runner.TEST_APC = DATA_DIR / 'test.apc'
runner.SUPPLEMENT_DIR = SUPPLEMENT_DIR
runner.SUPPLEMENT_FILES = [
    str(SUPPLEMENT_DIR / 'negative.tsv'),
    str(SUPPLEMENT_DIR / 'neutral.tsv'),
]
runner.ALL_CONFIGS = list(METHOD_CONFIGS)
runner.COMPACT_CONFIGS = []
runner.CONFIG_BY_ID = {cfg[7]: cfg for cfg in runner.ALL_CONFIGS}

# Kiểm tra: 5 config trong notebook phải trùng khớp định nghĩa gốc của repo.
assert set(runner.CONFIG_BY_ID) == set(METHODS), 'config_id không khớp METHODS'
assert len(runner.ALL_CONFIGS) == 5, f'Phải đúng 5 config, đang có {len(runner.ALL_CONFIGS)}'

args = [
    '--seeds', str(SEED),
    '--model-types', *APC_MODELS.keys(),
    '--configs', *METHODS.keys(),
    '--runs-dir', str(RUNS_DIR),
    '--resume',
]
# Ghép ATE theo seed khi ATE do notebook này huấn luyện; ngược lại dùng CSV dùng chung.
if USE_SEED_PAIRED_ATE:
    args += ['--ate-csv-dir', str(ATE_RUNS_DIR)]
else:
    args += ['--ate-csv', str(ATE_CSV)]

if torch.cuda.is_available():
    _free, _total = torch.cuda.mem_get_info()
    print(f'GPU      : {torch.cuda.get_device_name(0)}  '
          f'({torch.cuda.device_count()} thiết bị)  '
          f'VRAM trống {_free / 2**30:.2f} / {_total / 2**30:.2f} GB')
else:
    print('GPU      : (không có, chạy CPU)')
print('Dataset  :', DATA_DIR)
print('ATE CSV  :', ATE_CSV)
print('Output   :', RUNS_DIR)
print('Ma trận  :', len(APC_MODELS), 'backbone x', len(METHODS), 'method =', len(APC_MODELS) * len(METHODS), 'run')
for cfg in runner.ALL_CONFIGS:
    print(f'  {cfg[7]:<14} {cfg[6]:<12} lcf={cfg[0]} cdm={cfg[1]} tome={cfg[2]} resize={cfg[3]} merge={cfg[4]}')

In [ ]:
# Chạy 10 tổ hợp (5 method x 2 backbone). Chạy lại an toàn: --resume bỏ qua run đã xong.
sys.argv = ['run_multiseed.py', *args]
runner.main()

In [ ]:
import pandas as pd

raw = RUNS_DIR / 'results_raw.csv'
expected_runs = len(METHODS) * len(APC_MODELS)
if raw.is_file():
    df = pd.read_csv(raw)
    print(f'Hoàn thành: {len(df)} / {expected_runs} run')
    metric_columns = [
        'model_type', 'config_id', 'display_name', 'seed',
        'joint_precision', 'joint_recall', 'joint_f1',
        'joint_precision_macro', 'joint_recall_macro', 'joint_f1_macro',
        'joint_acc',
        'e2e_micro_precision', 'e2e_micro_recall', 'e2e_micro_f1',
        'e2e_macro_precision', 'e2e_macro_recall', 'e2e_macro_f1',
    ]
    available = [c for c in metric_columns if c in df.columns]
    # Sắp xếp theo đúng thứ tự 5 method đã chọn.
    order = {cid: i for i, cid in enumerate(METHODS)}
    table = df[available].copy()
    table['_o'] = table['config_id'].map(order)
    table = table.sort_values(['model_type', '_o']).drop(columns='_o')
    display(table)

    evaluation_csv = RUNS_DIR / 'evaluation_precision_recall_f1.csv'
    table.to_csv(evaluation_csv, index=False)
    print('Bảng đánh giá:', evaluation_csv)

    missing = [
        (m, c) for m in APC_MODELS for c in METHODS
        if not ((df['model_type'] == m) & (df['config_id'] == c)).any()
    ]
    if missing:
        print('Còn thiếu:', missing)
else:
    print('Chưa có kết quả:', raw)

# ── Liệt kê toàn bộ output ───────────────────────────────────────────────────
print('\n' + '=' * 72)
print('OUTPUT:', OUTPUT_ROOT)
print('=' * 72)
total = 0
for path in sorted(OUTPUT_ROOT.rglob('*')):
    if path.is_file():
        size = path.stat().st_size
        total += size
        print(f'  {size / 1024 / 1024:8.2f} MB  {path.relative_to(OUTPUT_ROOT)}')
print(f'\nTổng cộng: {total / 1024 / 1024:.1f} MB (giới hạn output Kaggle: 20 GB)')
print('\nFile kết quả chính:')
for name in ('results_raw.csv', 'results_aggregated.csv', 'results_summary.txt',
             'thesis_tables.txt', 'evaluation_precision_recall_f1.csv'):
    f = RUNS_DIR / name
    print(f'  {"OK " if f.is_file() else "-- "}{f}')